# 02 — Feature Engineering

Строит feature matrix (~105K строк × ~80 фичей) из 13M транзакций.  
Весь переиспользуемый код — в `src/features/`. Здесь только вызов и проверка.

In [1]:
import sys
sys.path.insert(0, '..')

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.features.build_features import build_feature_matrix
from src.config import FEATURE_MATRIX_PATH

## 1. Строим feature matrix

Если уже сохранена — загружаем из кеша. Иначе запускаем полный пайплайн (~5-10 мин на 13M транзакций).

In [2]:
if FEATURE_MATRIX_PATH.exists():
    print(f"Loading from cache: {FEATURE_MATRIX_PATH}")
    fm = pl.read_parquet(FEATURE_MATRIX_PATH)
else:
    fm = build_feature_matrix(verbose=True)

Loading parquet files...
  Transactions: 12,830,080  |  Cards: 105,000
Computing feature groups...
  Joining transactional (A/B/H/I)  (23 features)
  Joining mcc-based (C)  (10 features)
  Joining temporal (D)  (12 features)
  Joining recurring (E)  (6 features)
  Joining geo/channel (F/G)  (8 features)
  Joining graph (J)  (3 features)

Feature matrix: 105,000 cards × 64 columns
Saved → c:\Users\Yeska\OneDrive\Desktop\mdq-hackathon_1\notebooks\..\data\processed\feature_matrix.parquet


## 2. Базовая проверка (shape, nulls, label balance)

In [3]:
print(f"Shape: {fm.shape}")
print(f"\nLabel distribution:")
print(fm['label'].value_counts())

null_counts = {c: fm[c].null_count() for c in fm.columns if fm[c].null_count() > 0}
if null_counts:
    print(f"\nNull counts:")
    for col, cnt in sorted(null_counts.items(), key=lambda x: -x[1]):
        print(f"  {col}: {cnt}")
else:
    print("\nNo nulls ✓")

Shape: (105000, 64)

Label distribution:
shape: (2, 2)
┌───────┬───────┐
│ label ┆ count │
│ ---   ┆ ---   │
│ i8    ┆ u32   │
╞═══════╪═══════╡
│ 0     ┆ 80000 │
│ 1     ┆ 25000 │
└───────┴───────┘

No nulls ✓


## 3. Проверка топ-разделяющих фичей (медианы business vs consumer)

Ожидаемые кратности из EDA: b2b_spend_share 10.7×, night_recurring_share ∞, weekend_share 2.8×, median_ticket 6.5×

In [4]:
key_features = [
    'b2b_spend_share', 'night_recurring_share', 'weekend_share',
    'median_ticket_kzt', 'night_share', 'recurring_share',
    'merchant_hhi', 'business_hours_share', 'n_unique_mcc', 'lunch_dip_ratio',
]

biz = fm.filter(pl.col('label') == 1)
con = fm.filter(pl.col('label') == 0)

rows = []
for f in key_features:
    if f not in fm.columns:
        rows.append({'feature': f, 'business_median': 'MISSING', 'consumer_median': 'MISSING', 'ratio': None})
        continue
    b_med = biz[f].median()
    c_med = con[f].median()
    ratio = round(b_med / c_med, 2) if c_med and c_med != 0 else float('inf') if b_med else None
    rows.append({
        'feature': f,
        'business_median': round(b_med, 4) if b_med is not None else None,
        'consumer_median': round(c_med, 4) if c_med is not None else None,
        'ratio (b/c)': ratio,
    })

pl.DataFrame(rows)

feature,business_median,consumer_median,ratio (b/c)
str,f64,f64,f64
"""b2b_spend_share""",0.8114,0.0,inf
"""night_recurring_share""",0.1343,0.0,inf
"""weekend_share""",0.1241,0.3502,0.35
"""median_ticket_kzt""",84558.5,9673.5,8.74
"""night_share""",0.15,0.0538,2.79
"""recurring_share""",0.1343,0.0,inf
"""merchant_hhi""",0.2243,0.1023,2.19
"""business_hours_share""",0.6023,0.3366,1.79
"""n_unique_mcc""",15.0,32.0,0.47


## 4. Box plots топ-фичей

In [5]:
plot_feats = ['b2b_spend_share', 'night_recurring_share', 'weekend_share',
              'median_ticket_kzt', 'merchant_hhi', 'business_hours_share']

fm_pd = fm.select(plot_feats + ['label']).to_pandas()
fm_pd['label'] = fm_pd['label'].map({1: 'business', 0: 'consumer'})

fig = make_subplots(rows=2, cols=3, subplot_titles=plot_feats)
colors = {'business': '#636EFA', 'consumer': '#EF553B'}

for i, feat in enumerate(plot_feats):
    row, col = i // 3 + 1, i % 3 + 1
    for label, color in colors.items():
        vals = fm_pd[fm_pd['label'] == label][feat].dropna()
        fig.add_trace(
            go.Box(y=vals, name=label, marker_color=color,
                   showlegend=(i == 0), legendgroup=label),
            row=row, col=col
        )

fig.update_layout(height=600, title='Feature distributions: business vs consumer')
fig.show()

## 5. Корреляция фичей с таргетом

In [6]:
num_cols = [c for c in fm.columns
            if fm[c].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Int8, pl.Int16)
            and c != 'label']

corr_with_label = (
    fm.select(num_cols + ['label'])
    .to_pandas()
    .corr()['label']
    .drop('label')
    .sort_values(key=abs, ascending=False)
)

fig = px.bar(
    x=corr_with_label.values,
    y=corr_with_label.index,
    orientation='h',
    color=corr_with_label.values,
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title='Pearson correlation with label (business=1)',
    height=800,
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [7]:
print(f"Feature matrix ready: {fm.shape[0]:,} cards × {fm.shape[1]} columns")
print(f"Saved to: {FEATURE_MATRIX_PATH}")
print("\nNext: 03_model.ipynb — PU-Bagging + LightGBM")

Feature matrix ready: 105,000 cards × 64 columns
Saved to: c:\Users\Yeska\OneDrive\Desktop\mdq-hackathon_1\notebooks\..\data\processed\feature_matrix.parquet

Next: 03_model.ipynb — PU-Bagging + LightGBM
